# 09 — Git History Training Data

Mines real fix / refactor commits from ARO application repos and the ARO examples
directory to build training pairs grounded in actual before/after code changes.

Git logs are read **every time this notebook runs** — no upfront caching.

## Two pair types per commit hunk

| Type | User prompt | Assistant answer |
|------|-------------|------------------|
| **explanation** | Why was this ARO code changed? + old code | commit-message explanation |
| **fix** | Fix this ARO code (given explanation) + old code | new (fixed) code |

**Inputs:**
- `/Users/kris/Projects/ARO-Application/` — git history of ARO application repos
- `/Users/kris/Projects/ARO-App/Examples/` — git history of bundled examples
- `/Users/kris/Projects/ARO-App/Book/` and `Proposals/` — doc fix commits

**Output:**
- `git_applications_pairs.jsonl` — commit pairs from ARO-Application repos
- `git_examples_pairs.jsonl`     — commit pairs from Examples / Book / Proposals
- `git_transformation_pairs.jsonl` — transformation-style rewrite pairs
- All three files are appended to `../data/02_knowledge/knowledge_pairs.jsonl`

## Setup

In [1]:
import sys, importlib, json, re, subprocess
from pathlib import Path
from collections import defaultdict

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:           {ARO_ROOT}')
print(f'ARO-Application:    {ARO_APPLICATION_ROOT}')
print(f'Knowledge pairs:    {PAIRS_FILE}')

# ── System prompt for pair construction ──────────────────────────────────────
SYSTEM = (
    "You are an expert ARO programmer. ARO is a DSL where business features are expressed "
    "as Action-Result-Object statements inside named feature sets.\n"
    "Syntax: (Feature Name: Business Activity) { Action the <result> from the <object>. }\n"
    "Built-in actions: Extract, Retrieve, Compute, Validate, Compare, Create, Transform, "
    "Store, Update, Return, Emit, Publish, Log, Send, Start, Stop, Keepalive.\n"
    "Always respond with correct, idiomatic ARO code inside ```aro ... ``` fences."
)

# ── Commit filter patterns ────────────────────────────────────────────────────
FIX_PATTERN = re.compile(
    r'\b(fix|bug|patch|repair|correct|refactor|improve|update|rewrite|rename'
    r'|clean|simplify|optimis|optim|extract|move|add|remove|delete|replace'
    r'|adjust|change|tweak|rework|implement|support|handle|enable|disable)\b',
    re.IGNORECASE,
)

SYNTAX_CHANGE_PATTERN = re.compile(
    r'\b(syntax|language.?change|breaking.?change|deprecat|renamed?.?syntax'
    r'|grammar|keyword.?change|proposal|aro-\d{4})\b',
    re.IGNORECASE,
)

# Minimum non-whitespace lines in a diff hunk to be considered training-worthy
MIN_HUNK_LINES = 2


def _show_sample(pairs, n=2, label=''):
    import random as _r
    sample_pool = _r.sample(pairs, min(n, len(pairs)))
    print(f'\n── Sample ({label}, {len(pairs)} total) ──────────────────────')
    for i, s in enumerate(sample_pool, 1):
        if 'messages' in s:
            user = s['messages'][1]['content'] if len(s['messages']) > 1 else ''
            asst = s['messages'][2]['content'] if len(s['messages']) > 2 else ''
        else:
            user = s.get('instruction', s.get('user', ''))
            asst = s.get('output', s.get('assistant', ''))
        task = s.get('task_type', s.get('source', '?'))
        print(f'  [{i}] task={task}')
        print(f'       USER: {user[:120].strip()!r}')
        print(f'       ASST: {asst[:120].strip()!r}')
    print('─' * 60)

Fine-tuned model not found on HuggingFace, using base: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Fine-tuned model not found on HuggingFace, using base: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:           /Users/kris/Projects/ARO/ARO-Train
ARO-Application:    /Users/kris/Projects/ARO/ARO-Application
Knowledge pairs:    /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl


## Shared helpers

In [2]:
def git(*args, cwd):
    """Run a git command and return stdout, or '' on error."""
    result = subprocess.run(
        ['git'] + list(args),
        cwd=str(cwd), capture_output=True, text=True,
    )
    return result.stdout if result.returncode == 0 else ''


def get_fix_commits(repo_path: Path, file_glob: str = '*.aro', require_fix_pattern: bool = True) -> list[dict]:
    """
    Return a list of commit dicts for commits that touch `file_glob`.
    When require_fix_pattern=True (default), only commits whose message matches
    FIX_PATTERN are included.  Pass False for app repos where all .aro changes
    are worth training on regardless of commit message style.

    Each dict: {sha, subject, body, diff_text}
    """
    # One-liner log: sha TAB subject
    log = git(
        'log', '--format=%H\t%s\t%b<END_BODY>',
        '--diff-filter=M',      # only modified files (not pure adds/deletes)
        '--', file_glob,
        cwd=repo_path,
    )
    if not log.strip():
        return []

    commits = []
    for block in log.split('<END_BODY>'):
        block = block.strip()
        if not block:
            continue
        first_line, *rest = block.splitlines()
        parts = first_line.split('\t', 2)
        if len(parts) < 2:
            continue
        sha, subject = parts[0], parts[1]
        body = '\n'.join(rest).strip()

        if require_fix_pattern and not FIX_PATTERN.search(subject + ' ' + body):
            continue
        # Skip commits that document an ARO language/syntax change.
        # Training on these would teach the model that a prior syntax existed.
        if SYNTAX_CHANGE_PATTERN.search(subject + ' ' + body):
            continue

        diff_text = git(
            'show', sha,
            '--no-color', '--unified=3',
            '--', file_glob,
            cwd=repo_path,
        )
        if diff_text:
            commits.append({
                'sha': sha[:10],
                'subject': subject,
                'body': body,
                'diff_text': diff_text,
                'repo': repo_path.name,
            })

    return commits


def parse_diff_hunks(diff_text: str) -> list[dict]:
    """
    Parse a unified diff and return a list of hunks, each containing:
      {file, old_lines: [str], new_lines: [str]}
    Skips hunks that are pure whitespace / context-only changes.
    """
    hunks = []
    current_file = None
    old_lines, new_lines = [], []

    def _flush():
        nonlocal old_lines, new_lines
        o = [l for l in old_lines if l.strip()]
        n = [l for l in new_lines if l.strip()]
        if len(o) >= MIN_HUNK_LINES and len(n) >= MIN_HUNK_LINES:
            hunks.append({'file': current_file, 'old_lines': o, 'new_lines': n})
        old_lines, new_lines = [], []

    in_diff = False
    for line in diff_text.splitlines():
        if line.startswith('diff --git'):
            _flush()
            in_diff = True
            m = re.search(r'b/(.+)$', line)
            current_file = m.group(1) if m else None
            continue
        if not in_diff:
            continue
        if line.startswith('@@'):
            _flush()
            continue
        if line.startswith('---') or line.startswith('+++'):
            continue
        if line.startswith('-') and not line.startswith('---'):
            old_lines.append(line[1:])
        elif line.startswith('+') and not line.startswith('+++'):
            new_lines.append(line[1:])

    _flush()
    return hunks


def build_explanation(subject: str, body: str) -> str:
    """Compose a clean explanation string from commit subject + body."""
    parts = [subject.strip()]
    if body:
        # Trim noise lines (Git trailer lines, blank separators)
        body_lines = [
            l for l in body.splitlines()
            if l.strip() and not re.match(r'^(Co-authored|Signed-off|Reviewed)', l, re.I)
        ]
        if body_lines:
            parts.append('\n'.join(body_lines))
    return '\n\n'.join(parts)


def hunks_to_pairs(commit: dict) -> list[dict]:
    """
    Convert one commit into a list of training pair dicts.
    Returns [] if no usable hunks found.
    """
    explanation = build_explanation(commit['subject'], commit['body'])
    hunks = parse_diff_hunks(commit['diff_text'])
    pairs = []

    for hunk in hunks:
        old_code = '\n'.join(hunk['old_lines'])
        new_code = '\n'.join(hunk['new_lines'])
        fname    = hunk['file'] or 'main.aro'
        source   = f"{commit['repo']}/{fname}@{commit['sha']}"

        # --- pair type 1: old code → explanation why it was changed ---
        pairs.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM},
                {'role': 'user',      'content':
                    f'Why was this ARO code changed?\n\n'
                    f'```aro\n{old_code}\n```'},
                {'role': 'assistant', 'content': explanation},
            ],
            'task_type': 'debugging',
            'source': source,
        })

        # --- pair type 2: explanation + old code → new code ---
        pairs.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM},
                {'role': 'user',      'content':
                    f'{explanation}\n\n'
                    f'Fix this ARO code accordingly:\n\n'
                    f'```aro\n{old_code}\n```'},
                {'role': 'assistant', 'content':
                    f'```aro\n{new_code}\n```'},
            ],
            'task_type': 'debugging',
            'source': source,
        })

    return pairs



def hunks_to_transformation_pairs(commit: dict) -> list[dict]:
    """
    Convert one commit into raw transformation training pairs using the
    original commit message as the instruction.
    Returns [] if no usable hunks found.
    """
    subject = commit['subject']
    body    = commit.get('body', '')
    instruction = subject.strip()
    if body.strip():
        instruction += '\n\n' + body.strip()

    hunks = parse_diff_hunks(commit['diff_text'])
    pairs = []
    for hunk in hunks:
        old_code = '\n'.join(hunk['old_lines'])
        new_code = '\n'.join(hunk['new_lines'])
        fname    = hunk['file'] or 'main.aro'
        source   = f"{commit['repo']}/{fname}@{commit['sha']}"
        pairs.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM},
                {'role': 'user',      'content':
                    f'{instruction}\n\nApply this change to the following ARO code:\n\n'
                    f'```aro\n{old_code}\n```'},
                {'role': 'assistant', 'content': f'```aro\n{new_code}\n```'},
            ],
            'task_type': 'code_transformation',
            'instruction_type': 'commit_message',
            'source': source,
        })
    return pairs

print('Helpers loaded.')

Helpers loaded.


## Cell 1 — ARO-Application repos

Reads git history from every application in `/Users/kris/Projects/ARO-Application`.

In [3]:
app_pairs = []
app_stats = {}

app_repos = sorted(
    [d for d in ARO_APPLICATION_ROOT.iterdir() if d.is_dir() and (d / '.git').exists()]
)

print(f'Found {len(app_repos)} git repos under {ARO_APPLICATION_ROOT}\n')

for repo in app_repos:
    commits = get_fix_commits(repo, require_fix_pattern=False)
    pairs   = []
    for c in commits:
        pairs.extend(hunks_to_pairs(c))

    app_stats[repo.name] = {'commits': len(commits), 'pairs': len(pairs)}
    app_pairs.extend(pairs)
    print(f'  {repo.name:<25}  {len(commits):>3} qualifying commits  →  {len(pairs):>4} pairs')

print(f'\nTotal: {len(app_pairs)} training pairs')

# Write to JSONL
OUT_APPS = SCRIPT_DIR / 'git_applications_pairs.jsonl'
with OUT_APPS.open('w') as f:
    for p in app_pairs:
        f.write(json.dumps(p) + '\n')

print(f'Saved → {OUT_APPS}')
_show_sample(app_pairs, label='NB09 git_applications')

  mostrecentfile              10 qualifying commits  →    46 pairs
  rulpz                        0 qualifying commits  →     0 pairs
  sf                           5 qualifying commits  →    24 pairs

Total: 112 training pairs
Saved → /Users/kris/Projects/ARO/ARO-Train/Train/script/git_applications_pairs.jsonl

── Sample (NB09 git_applications, 112 total) ──────────────────────
  [1] task=debugging
       USER: 'Why was this ARO code changed?\n\n```aro\n    (* Stat each file and store directly *)\n        Store the <stats> into the <f'
       ASST: 'Skip directories when storing file stats — only track regular files\n\ndirectories (including .git) are never added as can'
  [2] task=debugging
       USER: "Simplify: drop manual counter, use repository's own count in ProcessStats\n\nthe observer not being able to read file-stat"
       ASST: '```aro\n    (* Only react to new entries — deletions must not re-trigger the cascade *)\n            Emit a <ProcessStats:'
──────────────────────

## Cell 2 — ARO repo examples & runtime

Reads fix commits from the ARO repository itself — both `Examples/` `.aro` files
and `Sources/` Swift files that contain inline ARO snippets in comments or strings.

In [4]:
aro_pairs = []
aro_stats = {}

# Sub-paths inside ARO_ROOT to mine; each entry is (label, glob)
ARO_TARGETS = [
    ('Examples', 'Examples/**/*.aro'),
    # Book commits are markdown diffs, not ARO code — excluded
    # Proposals commits are spec text, not ARO code — excluded
]

print(f'Mining ARO repo: {ARO_ROOT}\n')

for label, glob in ARO_TARGETS:
    commits = get_fix_commits(ARO_ROOT, glob)
    # Override repo name so source tags are descriptive
    for c in commits:
        c['repo'] = f'aro/{label}'
    pairs = []
    for c in commits:
        pairs.extend(hunks_to_pairs(c))

    aro_stats[label] = {'commits': len(commits), 'pairs': len(pairs)}
    aro_pairs.extend(pairs)
    print(f'  {label:<12}  {len(commits):>3} qualifying commits  →  {len(pairs):>4} pairs')

print(f'\nTotal: {len(aro_pairs)} training pairs')

# Write to JSONL
OUT_ARO = SCRIPT_DIR / 'git_examples_pairs.jsonl'
with OUT_ARO.open('w') as f:
    for p in aro_pairs:
        f.write(json.dumps(p) + '\n')

print(f'Saved → {OUT_ARO}')
_show_sample(aro_pairs, label='NB09 git_examples')

  Examples       50 qualifying commits  →   110 pairs

Total: 110 training pairs
Saved → /Users/kris/Projects/ARO/ARO-Train/Train/script/git_examples_pairs.jsonl

── Sample (NB09 git_examples, 110 total) ──────────────────────
  [1] task=debugging
       USER: 'Why was this ARO code changed?\n\n```aro\n(* MarkdownRenderer - Demonstrates using a managed Python plugin for Markdown pro'
       ASST: 'Fix dynamic plugin action dispatch and parameter passing\n\n- Check dynamicHandlers after built-in actions in FeatureSetEx'
  [2] task=debugging
       USER: 'Why was this ARO code changed?\n\n```aro\n    <Extract> the <data> from the <request: body>.\n    <Create> the <user> with <'
       ASST: 'refactor: replace generic variable names with meaningful names\n\ncontext-specific names throughout all examples.\nChanges'
────────────────────────────────────────────────────────────


## Cell 3 — Code Transformation pairs

Mines **every** commit that touches `.aro` files from both ARO `Examples/` and all ARO-Application repos, regardless of commit message style.  
Produces two kinds of training pairs per hunk:

1. **Raw** — original commit message as instruction
2. **Enhanced** — a better instruction generated by the model from the message + diff

In [5]:
# ── Mine raw transformation pairs (no FIX_PATTERN filter) ──────────────
transform_pairs_raw = []
transform_stats     = {}

# ARO-Application repos
print(f'Mining ARO-Application repos for transformation pairs...\n')
for repo in app_repos:
    commits = get_fix_commits(repo, require_fix_pattern=False)
    pairs   = []
    for c in commits:
        pairs.extend(hunks_to_transformation_pairs(c))
    transform_stats[repo.name] = {'commits': len(commits), 'raw_pairs': len(pairs)}
    transform_pairs_raw.extend(pairs)
    print(f'  {repo.name:<25}  {len(commits):>3} commits  →  {len(pairs):>4} raw pairs')

# ARO Examples
print(f'\nMining ARO Examples...')
for label, glob in [('Examples', 'Examples/**/*.aro')]:
    commits = get_fix_commits(ARO_ROOT, glob, require_fix_pattern=False)
    for c in commits:
        c['repo'] = f'aro/{label}'
    pairs = []
    for c in commits:
        pairs.extend(hunks_to_transformation_pairs(c))
    transform_stats[f'aro/{label}'] = {'commits': len(commits), 'raw_pairs': len(pairs)}
    transform_pairs_raw.extend(pairs)
    print(f'  {label:<25}  {len(commits):>3} commits  →  {len(pairs):>4} raw pairs')

print(f'\nTotal raw transformation pairs: {len(transform_pairs_raw)}')


  Examples                    54 commits  →    58 raw pairs

Total raw transformation pairs: 114


### Generate enhanced instructions

For each raw pair, ask the model to rewrite the commit message as a clear, imperative instruction that describes the code change.
Both the raw and enhanced versions are saved as separate training samples.

In [6]:
# Load model (reuse if already loaded in this kernel session)
try:
    _t9_model
    print('Model already loaded.')
except NameError:
    _t9_model, _t9_tok, _, _t9_gen, _t9_sampler_fn = load_model(with_adapter=False)

def _t9_chat(messages, max_tokens=120):
    from mlx_lm import generate as _gen_fn
    prompt = _t9_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return _t9_gen(_t9_model, _t9_tok, prompt=prompt, max_tokens=max_tokens,
                   verbose=False).strip()


Model ready.


In [7]:
_ENHANCE_SYSTEM = (
    "You are a technical writing assistant for the ARO programming language. "
    "Given a git commit message and an ARO code diff, write a single concise "
    "imperative instruction (1-2 sentences, no jargon) that tells a developer "
    "exactly what change to make. Start with a verb. Do not mention git or commits. "
    "IMPORTANT: treat the code in the 'New' block as the ONLY correct ARO syntax. "
    "Never reference 'old syntax', 'new syntax', 'migration', or language version changes. "
    "Frame the instruction as a best-practice improvement, not a syntax migration."
)

def generate_enhanced_instruction(subject, body, old_code, new_code):
    diff_summary = (
        f"Old:\n```aro\n{old_code[:600]}\n```\n"
        f"New:\n```aro\n{new_code[:600]}\n```"
    )
    commit_text = subject.strip()
    if body.strip():
        commit_text += "\n" + body.strip()
    messages = [
        {'role': 'system', 'content': _ENHANCE_SYSTEM},
        {'role': 'user',   'content':
            f"Commit message: {commit_text}\n\n{diff_summary}\n\n"
            f"Write the instruction:"},
    ]
    return _t9_chat(messages, max_tokens=120)


# ── Build enhanced pairs ─────────────────────────────────────────────────
transform_pairs_enhanced = []

# Collect unique (commit, hunk) tuples to avoid duplicate model calls
_seen_sources = set()
_source_to_enhanced = {}

print(f'Generating enhanced instructions for {len(transform_pairs_raw)} pairs...')
for i, pair in enumerate(transform_pairs_raw):
    src = pair.get('source','')
    user_msg = pair['messages'][1]['content']
    # Extract old/new from messages
    old_code = pair['messages'][1]['content']
    new_code = pair['messages'][2]['content']
    # Strip fences
    old_bare = re.sub(r'```aro\n|```', '', old_code.split("Apply this change")[1] if "Apply this change" in old_code else old_code).strip()
    new_bare = re.sub(r'```aro\n|```', '', new_code).strip()
    # Original instruction = everything before "\n\nApply this change"
    raw_instr = old_code.split("\n\nApply this change")[0] if "Apply this change" in old_code else ""
    subject_line = raw_instr.splitlines()[0] if raw_instr else ""
    body_rest    = "\n".join(raw_instr.splitlines()[2:]) if raw_instr else ""

    if src not in _source_to_enhanced:
        enhanced = generate_enhanced_instruction(subject_line, body_rest, old_bare, new_bare)
        _source_to_enhanced[src] = enhanced
        if (i + 1) % 20 == 0 or i == 0:
            print(f'  [{i+1}/{len(transform_pairs_raw)}] {src[:60]}')
            print(f'    raw:      {subject_line[:80]}')
            print(f'    enhanced: {enhanced[:100]}')
    else:
        enhanced = _source_to_enhanced[src]

    transform_pairs_enhanced.append({
        'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content':
                f'{enhanced}\n\nApply this change to the following ARO code:\n\n'
                f'```aro\n{old_bare}\n```'},
            {'role': 'assistant', 'content': f'```aro\n{new_bare}\n```'},
        ],
        'task_type': 'code_transformation',
        'instruction_type': 'enhanced',
        'source': src,
    })

print(f'\nEnhanced pairs: {len(transform_pairs_enhanced)}')



Enhanced pairs: 114


In [8]:
# ── Save transformation pairs ────────────────────────────────────────────
OUT_TRANSFORM = SCRIPT_DIR / 'git_transformation_pairs.jsonl'
all_transform = transform_pairs_raw + transform_pairs_enhanced

with OUT_TRANSFORM.open('w') as f:
    for p in all_transform:
        f.write(json.dumps(p) + '\n')

print(f'Saved {len(all_transform)} transformation pairs → {OUT_TRANSFORM}')
print(f'  raw pairs      : {len(transform_pairs_raw)}')
print(f'  enhanced pairs : {len(transform_pairs_enhanced)}')

_show_sample(all_transform, label='NB09 git_transformation')

Saved 228 transformation pairs → /Users/kris/Projects/ARO/ARO-Train/Train/script/git_transformation_pairs.jsonl
  raw pairs      : 114
  enhanced pairs : 114

── Sample (NB09 git_transformation, 228 total) ──────────────────────
  [1] task=code_transformation
       USER: 'Use markdown specifier for HTML-to-Markdown conversion\n\n- Update storage.aro to save .md files with frontmatter\n- Update'
       ASST: '```aro\n    (* Extract markdown content from HTML using ParseHtml action *)\n    <ParseHtml> the <markdown-result: markdow'
  [2] task=code_transformation
       USER: 'Replace the range loop iteration with repository-based iteration using the index operator to access elements directly by'
       ASST: '```aro\nCompute the <next-ridx> from <ridx> + 1.\n        Update the <rctr: val> with <next-ridx>.\n        Store the <rctr'
────────────────────────────────────────────────────────────


## Inspect samples

In [9]:
import random

all_pairs = app_pairs + aro_pairs
sample = random.sample(all_pairs, min(3, len(all_pairs)))

for i, p in enumerate(sample):
    print(f'── Sample {i+1}  [{p.get("source","")}] ──────────────────────────────────')
    for msg in p['messages']:
        if msg['role'] == 'system':
            continue
        role = msg['role'].upper()
        print(f'[{role}]\n{msg["content"][:400]}')
        if len(msg['content']) > 400:
            print('...')
        print()

── Sample 1  [aro/Examples/Examples/UserService/events.aro@ac9a6817ba] ──────────────────────────────────
[USER]
Why was this ARO code changed?

```aro
(* Handler 1: Log when a user is updated via API *)
(* Handler 2: Write user data to file when updated via API *)
```

[ASSISTANT]
fix(runtime): correct PUT update behavior and improve Merge action

- Add merge verbs to FeatureSetExecutor's execution whitelist
- Fix RepositoryStorage to update by id instead of appending duplicates
- Add UserCreated Handler to write files on user creation
- Update ActionsReference.md with proper Merge documentation
- Rebuild website with updated documentation
The PUT update was creating duplic
...

── Sample 2  [StatusPost/websocket.aro@4ba5f159bc] ──────────────────────────────────
[USER]
Prometheus Metrics

Fix this ARO code accordingly:

```aro
    <Extract> the <connection-id> from the <event: id>.
    <Log> "WebSocket client connected" to the <console>.
    <Return> an <OK: status> for the <connecti

## Append to knowledge_pairs.jsonl

Appends both JSONL files to the pipeline's knowledge pairs so the next fine-tune run
includes this data.  Run this cell only when you want to incorporate the pairs.

In [10]:
PAIRS_FILE.parent.mkdir(parents=True, exist_ok=True)

clean_notebook_pairs('NB09')

total_appended = 0
for path in (OUT_APPS, OUT_ARO, OUT_TRANSFORM):
    lines = [l for l in path.read_text().splitlines() if l.strip()]
    pairs = []
    for line in lines:
        pair = json.loads(line)
        pair['notebook'] = 'NB09'
        pairs.append(pair)
    save_notebook_pairs('NB09', pairs)
    total_appended += len(pairs)
    print(f'  {path.name:<35} {len(pairs)} pairs appended')

print(f'\nTotal appended to {PAIRS_FILE}: {total_appended}')

  git_applications_pairs.jsonl        112 pairs appended
  git_examples_pairs.jsonl            110 pairs appended
  git_transformation_pairs.jsonl      228 pairs appended

Total appended to /Users/kris/Projects/ARO/ARO-Train/Train/data/02_knowledge/knowledge_pairs.jsonl: 450


## Summary

In [11]:
print('═' * 60)
print('Git History Training Data — Summary')
print('═' * 60)

print('\nARO-Application repos:')
for name, s in sorted(app_stats.items()):
    print(f'  {name:<25}  {s["commits"]:>3} commits  {s["pairs"]:>4} pairs')

print('\nARO repo:')
for name, s in sorted(aro_stats.items()):
    print(f'  {name:<25}  {s["commits"]:>3} commits  {s["pairs"]:>4} pairs')

print(f'\nTotal pairs : {len(app_pairs) + len(aro_pairs)}')
print(f'  {OUT_APPS.name}: {len(app_pairs)}')
print(f'  {OUT_ARO.name}: {len(aro_pairs)}')
print('\nTransformation pairs by source:')
for name, s in sorted(transform_stats.items()):
    enhanced = sum(1 for p in transform_pairs_enhanced if p.get('source', '').startswith(name.split('/')[0]))
    print(f'  {name:<25}  {s["commits"]:>3} commits  {s["raw_pairs"]:>4} raw  +{enhanced:>4} enhanced')

print(f'\nTransformation pairs : {len(transform_pairs_raw)} raw + {len(transform_pairs_enhanced)} enhanced = {len(transform_pairs_raw)+len(transform_pairs_enhanced)} total')
print(f'  {OUT_TRANSFORM.name}: {len(all_transform)}')


════════════════════════════════════════════════════════════
Git History Training Data — Summary
════════════════════════════════════════════════════════════

ARO-Application repos:
  Crawler                      8 commits    22 pairs
  FediTerm                     2 commits     0 pairs
  StatusPost                   5 commits     6 pairs
  Sumup                        4 commits    12 pairs
  SystemLoadMonitor            0 commits     0 pairs
  homebrew-applications        0 commits     0 pairs
  mm                           4 commits     2 pairs
  mostrecentfile              10 commits    46 pairs
  rulpz                        0 commits     0 pairs
  sf                           5 commits    24 pairs

ARO repo:
  Examples                    50 commits   110 pairs

Total pairs : 222
  git_applications_pairs.jsonl: 112
  git_examples_pairs.jsonl: 110

Transformation pairs by source:
  Crawler                      8 commits    11 raw  +  11 enhanced
  FediTerm                     2 comm

In [12]:
# ── Samples per category ─────────────────────────────────────────────────────
import json, random

_app_pairs, _aro_pairs = [], []
for path, target in ((OUT_APPS, _app_pairs), (OUT_ARO, _aro_pairs)):
    if path.exists():
        with open(path) as f:
            for line in f:
                if line.strip():
                    target.append(json.loads(line))

def _show_git_samples(pairs, label, n=2):
    if not pairs:
        print(f'  (no pairs in {label})')
        return
    # Split into explanation vs fix by looking at user message content
    explanation = [p for p in pairs if 'Why was this' in p['messages'][1]['content']]
    fix         = [p for p in pairs if p not in explanation]
    for sub_label, pool in [('explanation', explanation), ('fix', fix)]:
        if not pool:
            continue
        print(f'\n{"─"*72}')
        print(f'  {label} → {sub_label}  ({len(pool)} pairs)')
        print('─'*72)
        for s in random.sample(pool, min(n, len(pool))):
            user_msg = s['messages'][1]['content']
            asst_msg = s['messages'][2]['content']
            print(f'[{s.get("source","")}]')
            print(f'Q: {user_msg[:240]}{"..." if len(user_msg) > 240 else ""}')
            print(f'A: {asst_msg[:280]}{"..." if len(asst_msg) > 280 else ""}')
            print()

_show_git_samples(_app_pairs, 'ARO-Application repos')
_show_git_samples(_aro_pairs, 'ARO repo (Examples/Book/Proposals)')


────────────────────────────────────────────────────────────────────────
  ARO-Application repos → explanation  (56 pairs)
────────────────────────────────────────────────────────────────────────
[mostrecentfile/processor.aro@75f6d067d7]
Q: Why was this ARO code changed?

```aro
(* Processor — Three responsibilities:
      the count has reached the threshold, then prunes to the single most-
      recently-modified entry and doubles the threshold (capped at 1024).
   3. Finaliz...
A: Extract shared pruning logic into PruneStats handler

duplicating the 20-line tracker/winner/delete loop. PruneStats Handler
owns the logic once.

[mm/main.aro@36bbfc8bfc]
Q: Why was this ARO code changed?

```aro
        body: { "channel_id": <channel-id>, "message": <message> }
    Log "Sent to " ++ <recipient> to the <console>.
```
A: ⏺ Clean (the parameter warning is expected — it's a framework-injected object). Good to go. The openapi.yaml documents     the parameter schema and main.aro now fetches bo

In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
# ── Final status: git training pairs per source ───────────────────────────────
import numpy as np
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '09_git_training.png'

# Merge app_stats and aro_stats into one list for plotting
_sources = []
_commits = []
_pairs   = []

for name, s in sorted(app_stats.items()):
    _sources.append(name.replace('-', '\n'))
    _commits.append(s['commits'])
    _pairs.append(s['pairs'])

for name, s in sorted(aro_stats.items()):
    _sources.append(f'aro/\n{name}')
    _commits.append(s['commits'])
    _pairs.append(s['pairs'])

x = np.arange(len(_sources))
w = 0.38

fig, ax = plt.subplots(figsize=(max(10, len(_sources) * 1.1), 5))
b1 = ax.bar(x - w/2, _commits, w, label='Commits', color='#85c1e9', edgecolor='white')
b2 = ax.bar(x + w/2, _pairs,   w, label='Pairs',   color='#2ecc71', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(_sources, fontsize=8)
ax.set_ylabel('Count')
ax.set_title(
    f'Git History Training — {sum(_pairs):,} pairs from {sum(_commits)} commits',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
for bar in b2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3, str(int(h)),
                ha='center', va='bottom', fontsize=7, color='#333')
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


Saved: run/2026-03-29/09_git_training.png
